In [74]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
sys.path.append('/Users/kevinlaventure/python_code')
from python_module.pricing_model import SABRModel
from scipy.optimize import minimize, LinearConstraint

# Configure pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:_.2f}')

In [103]:
def compute_option_surface(
    F: float, 
    K_list: list, 
    T_list: list, 
    alpha_list: list, 
    beta: float, 
    rho: float, 
    nu: float,
    r: float, 
    slide_scenario=None,
    slide_type: str = 'spot_only', 
    slide_compute: str = 'option_pnl',
    compute_bs_greeks: bool = True, 
    compute_model_greek: bool = False
) -> pd.DataFrame:
    """
    Computes SABR option prices and Greeks over a grid of strikes and maturities.
    
    Args:
        F: Forward price
        K_list: List of strike prices
        T_list: List of times to maturity (in years)
        alpha_list: List of alpha (volatility) parameters
        beta, rho, nu: SABR parameters
        r: Risk-free rate
        slide_scenario: List of spot bumps (optional)
        slide_type: 'spot_vol' or 'spot_only'
        slide_compute: PnL calculation type ('delta_hedged_pnl', 'option_pnl', 'delta_pnl')
        compute_bs_greeks: If True, returns Black-Scholes Greeks
        compute_model_greek: If True, returns SABR model Greeks
        
    Returns:
        DataFrame with rows for each (K, T, alpha) combination containing:
        - Input parameters: F, K, T, alpha, beta, rho, nu, r, option_type
        - IV: Implied volatility
        - price: Option price
        - Greeks: delta, gamma, vega, theta, vanna, volga
        - Model Greeks (if compute_model_greek=True): sabr_delta, sabr_gamma, sabr_vega, sabr_vanna, sabr_volga, sabr_theta
        - Slides: PnL or price differences for each slide scenario
        
    Note:
        Option type is determined automatically: call if K > F, put if K ≤ F
    """
    results = []
    
    for i in range(len(alpha_list)):
        alpha = alpha_list[i]
        T = T_list[i]
        for K in K_list:
            # Determine option type: call if K > F, put otherwise
            option_type = 'call' if K > F else 'put'
            
            result = SABRModel.compute_option(
                F=F, 
                K=K, 
                T=T, 
                alpha=alpha, 
                beta=beta, 
                rho=rho, 
                nu=nu,
                r=r, 
                option_type=option_type, 
                slide_scenario=slide_scenario,
                slide_type=slide_type, 
                slide_compute=slide_compute,
                compute_bs_greeks=compute_bs_greeks, 
                compute_model_greek=compute_model_greek)
                
            # Build row with inputs and results
            row = {
                'F': F,
                'K': K,
                'T': T,
                'alpha': alpha,
                'beta': beta,
                'rho': rho,
                'nu': nu,
                'r': r,
                'option_type': option_type,
            }
            
            # Add all result fields
            row.update(result)
            
            results.append(row)
    
    # Convert to DataFrame
    df = pd.DataFrame(results)
    
    # Reorder columns: inputs first, then IV and price, then greeks, then slides
    input_cols = ['F', 'K', 'T', 'alpha', 'beta', 'rho', 'nu', 'r', 'option_type']
    price_cols = ['IV', 'price']
    greek_cols = ['delta', 'gamma', 'vega', 'theta', 'vanna', 'volga']
    sabr_greek_cols = ['sabr_delta', 'sabr_gamma', 'sabr_vega', 'sabr_vanna', 'sabr_volga', 'sabr_theta']
    
    # Build column order
    col_order = input_cols + price_cols
    col_order += [c for c in greek_cols if c in df.columns]
    col_order += [c for c in sabr_greek_cols if c in df.columns]
    
    # Add slide columns (remaining columns)
    slide_cols = [c for c in df.columns if c not in col_order]
    col_order += slide_cols
    
    # Reorder dataframe
    df = df[col_order]
    
    return df

In [105]:
# Generate surface with slides
F = 100.0  # Forward price
K_list = [90, 95, 100, 105, 110]  # Strike prices
K_list = np.linspace(70, 130, 60)  # 9 strikes from 80 to 120 --- IGNORE ---
T_list = [0.25]  # Times to maturity
alpha_list = [0.1]  # Alpha (volatility) parameters

beta = 1
rho = -0.9
nu = 2
r = 0.0

# Compute option surface with slides
slide_scenario = [-0.3, -0.1, -0.05, -0.04, -0.03, -0.02, -0.01, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.3]
df_surface = compute_option_surface(
    F=F,
    K_list=K_list,
    T_list=T_list,
    alpha_list=alpha_list,
    beta=beta,
    rho=rho,
    nu=nu,
    r=r,
    slide_scenario=slide_scenario,
    compute_bs_greeks=True,
    compute_model_greek=False
)
df_surface['symbol'] = df_surface['K'].astype(str) + '_' + df_surface['T'].astype(str)
df_surface.set_index('symbol', inplace=True)
df_ =  df_surface.loc[:, slide_scenario + ['theta']]